### HF形式->ESM形式変換(lm_headは除外)

In [2]:
import os
import re
import torch
from transformers import AutoModel
import esm

# ====== 設定 ======
HF_DIR = "/data3/taihei/matsunaga-repos/vhh_up/PLM/models/best_models/esm2_8m_ssft-sft_251201/encoder_lr_5e-4_batch_size_32_encoder_weight_decay_0.01/encoder"
OUT_PT = "/data3/taihei/matsunaga-repos/vhh_up/PLM/models/best_models/esm2_8m_ssft-sft_251201/encoder_lr_5e-4_batch_size_32_encoder_weight_decay_0.01/encoder/converted_model.pt"
ESM_NAME = "esm2_t6_8M_UR50D"
ADD_ESM_PREFIX = True                # SFT_hot.pt に合わせて 'esm.' を付けて保存
# ★ 重要: lm_head.* は保存しない（補完もしない）
FILL_LM_HEAD_FROM_ESM_DEFAULT = False
# ====================

# safetensors 側キー（例）
# embeddings.word_embeddings.weight
# embeddings.position_embeddings.weight         -> (ESM側に対応なし: rotary使用のため破棄)
# encoder.emb_layer_norm_after.{weight,bias}    -> esm.emb_layer_norm_after.{weight,bias}
# encoder.layer.N.attention.self.{query,key,value}.{weight,bias}
# encoder.layer.N.attention.output.dense.{weight,bias}
# encoder.layer.N.attention.LayerNorm.{weight,bias}
# encoder.layer.N.intermediate.dense.{weight,bias}
# encoder.layer.N.output.dense.{weight,bias}
# encoder.layer.N.LayerNorm.{weight,bias}
# encoder.layer.N.attention.self.rotary_embeddings.inv_freq
# pooler.dense.{weight,bias}                    -> head.0.{weight,bias}（SFT_hot.pt 譲り）
# contact_head.regression.{weight,bias}         -> esm.contact_head.regression.{weight,bias}

LAYER_RE = re.compile(r"^encoder\.layer\.(\d+)\.")

def map_hf_key_to_esm(hf_key: str):
    """HF(.safetensors) -> ESM(.pt) の確定マッピング。"""
    m = LAYER_RE.match(hf_key)
    layer = int(m.group(1)) if m else None

    # embeddings
    if hf_key == "embeddings.word_embeddings.weight":
        return "esm.embed_tokens.weight", True
    if hf_key == "embeddings.position_embeddings.weight":
        return None, False  # rotaryなので破棄

    # emb_layer_norm_after
    if hf_key == "encoder.emb_layer_norm_after.weight":
        return "esm.emb_layer_norm_after.weight", True
    if hf_key == "encoder.emb_layer_norm_after.bias":
        return "esm.emb_layer_norm_after.bias", True

    # contact head
    if hf_key == "contact_head.regression.weight":
        return "esm.contact_head.regression.weight", True
    if hf_key == "contact_head.regression.bias":
        return "esm.contact_head.regression.bias", True

    # pooler -> head.0（SFT_hot.pt 準拠、'esm.' は付けない）
    if hf_key == "pooler.dense.weight":
        return "head.0.weight", True
    if hf_key == "pooler.dense.bias":
        return "head.0.bias", True

    # 各層
    if layer is not None:
        if hf_key.endswith(".attention.self.rotary_embeddings.inv_freq"):
            return f"esm.layers.{layer}.self_attn.rot_emb.inv_freq", True
        if hf_key.endswith(".attention.self.query.weight"):
            return f"esm.layers.{layer}.self_attn.q_proj.weight", True
        if hf_key.endswith(".attention.self.query.bias"):
            return f"esm.layers.{layer}.self_attn.q_proj.bias", True
        if hf_key.endswith(".attention.self.key.weight"):
            return f"esm.layers.{layer}.self_attn.k_proj.weight", True
        if hf_key.endswith(".attention.self.key.bias"):
            return f"esm.layers.{layer}.self_attn.k_proj.bias", True
        if hf_key.endswith(".attention.self.value.weight"):
            return f"esm.layers.{layer}.self_attn.v_proj.weight", True
        if hf_key.endswith(".attention.self.value.bias"):
            return f"esm.layers.{layer}.self_attn.v_proj.bias", True
        if hf_key.endswith(".attention.output.dense.weight"):
            return f"esm.layers.{layer}.self_attn.out_proj.weight", True
        if hf_key.endswith(".attention.output.dense.bias"):
            return f"esm.layers.{layer}.self_attn.out_proj.bias", True
        if hf_key.endswith(".attention.LayerNorm.weight"):
            return f"esm.layers.{layer}.self_attn_layer_norm.weight", True
        if hf_key.endswith(".attention.LayerNorm.bias"):
            return f"esm.layers.{layer}.self_attn_layer_norm.bias", True
        if hf_key.endswith(".intermediate.dense.weight"):
            return f"esm.layers.{layer}.fc1.weight", True
        if hf_key.endswith(".intermediate.dense.bias"):
            return f"esm.layers.{layer}.fc1.bias", True
        if hf_key.endswith(".output.dense.weight"):
            return f"esm.layers.{layer}.fc2.weight", True
        if hf_key.endswith(".output.dense.bias"):
            return f"esm.layers.{layer}.fc2.bias", True
        if hf_key.endswith(".LayerNorm.weight"):
            return f"esm.layers.{layer}.final_layer_norm.weight", True
        if hf_key.endswith(".LayerNorm.bias"):
            return f"esm.layers.{layer}.final_layer_norm.bias", True

    return None, False


def main():
    # --- HFロード ---
    print(f"Load HF (safetensors) from: {HF_DIR}")
    hf_model = AutoModel.from_pretrained(HF_DIR, trust_remote_code=True)
    hf_sd = hf_model.state_dict()
    print(f"HF state_dict params: {len(hf_sd)}")

    # --- ESM 本家モデル（参照取得 & 検証用）---
    print(f"Load ESM model: {ESM_NAME}")
    esm_model, _ = getattr(esm.pretrained, ESM_NAME)()
    esm_sd_ref = esm_model.state_dict()

    # --- 変換 ---
    out_sd = {}
    dropped, shape_mismatch = [], []
    mapped_cnt = 0

    for k, v in hf_sd.items():
        out_key, keep = map_hf_key_to_esm(k)
        if not keep:
            dropped.append(k)
            continue
        # 形状チェック（参照がある場合）
        if out_key.startswith("esm.") and (out_key[4:] in esm_sd_ref):
            if esm_sd_ref[out_key[4:]].shape != v.shape:
                shape_mismatch.append((k, out_key, tuple(v.shape), tuple(esm_sd_ref[out_key[4:]].shape)))
                continue
        out_sd[out_key] = v.clone()
        mapped_cnt += 1

    print(f"Mapped params: {mapped_cnt}")
    if dropped:
        print(f"Dropped (no ESM counterpart) count: {len(dropped)}")
        ex = [x for x in dropped if "position_embeddings" in x]
        if ex:
            print("  includes:", ex[:1], " ...")

    if shape_mismatch:
        print("Shape mismatches:")
        for i, (src, dst, s1, s2) in enumerate(shape_mismatch[:10], 1):
            print(f"  {i:02d}. {src} -> {dst}  HF={s1}, ESM={s2}")
        print("Aborting due to shape mismatch.")
        return

    # --- ★ lm_head.* は保存しない（完全除外） ---
    if FILL_LM_HEAD_FROM_ESM_DEFAULT:
        print("WARNING: FILL_LM_HEAD_FROM_ESM_DEFAULT=True is not allowed in this script.")
    # 念のため、存在していた場合も除外
    out_sd = {k: v for k, v in out_sd.items()
              if not (k.startswith("esm.lm_head.") or k.startswith("lm_head."))}

    # --- rotary inv_freq 不足分の補完（安全策） ---
    for name in list(esm_sd_ref.keys()):
        if name.endswith(".rot_emb.inv_freq"):
            out_k = ("esm." + name) if ADD_ESM_PREFIX else name
            if out_k not in out_sd:
                out_sd[out_k] = esm_sd_ref[name].clone()

    # --- 保存 ---
    os.makedirs(os.path.dirname(OUT_PT), exist_ok=True)
    torch.save(out_sd, OUT_PT)
    print(f"\n✅ Saved ESM-compatible state_dict (no lm_head) to: {OUT_PT}")

    # --- 検証：ESM にロード（'esm.' を剥がし、head.* は渡さない） ---
    test_sd = {}
    for k, v in out_sd.items():
        if k.startswith("esm."):
            key = k[4:]
            if not key.startswith("lm_head."):  # 念のため
                test_sd[key] = v

    missing, unexpected = esm_model.load_state_dict(test_sd, strict=False)
    print(f"\n🔎 Verify load into ESM (missing={len(missing)}, unexpected={len(unexpected)})")
    if missing:
        for i, m in enumerate(missing[:20], 1):
            print(f"  {i:02d}. {m}")
    if unexpected:
        for i, u in enumerate(unexpected[:20], 1):
            print(f"  U{i:02d}. {u}")

    # --- 追加チェック：埋め込みが渡した値と一致するか ---
    if "embed_tokens.weight" in test_sd:
        delta = (esm_model.embed_tokens.weight.detach().cpu() - test_sd["embed_tokens.weight"].detach().cpu()).abs().max().item()
        print(f"\nmax|Δ| after load (embed_tokens.weight) = {delta:.6g}  # ≈0 を期待")

    print("\nDone.")

if __name__ == "__main__":
    main()

Load HF (safetensors) from: /data3/taihei/matsunaga-repos/vhh_up/PLM/models/best_models/esm2_8m_ssft-sft_251201/encoder_lr_5e-4_batch_size_32_encoder_weight_decay_0.01/encoder
HF state_dict params: 110
Load ESM model: esm2_t6_8M_UR50D
Mapped params: 109
Dropped (no ESM counterpart) count: 1
  includes: ['embeddings.position_embeddings.weight']  ...

✅ Saved ESM-compatible state_dict (no lm_head) to: /data3/taihei/matsunaga-repos/vhh_up/PLM/models/best_models/esm2_8m_ssft-sft_251201/encoder_lr_5e-4_batch_size_32_encoder_weight_decay_0.01/encoder/converted_model.pt

🔎 Verify load into ESM (missing=6, unexpected=0)
  01. lm_head.weight
  02. lm_head.bias
  03. lm_head.dense.weight
  04. lm_head.dense.bias
  05. lm_head.layer_norm.weight
  06. lm_head.layer_norm.bias

max|Δ| after load (embed_tokens.weight) = 0  # ≈0 を期待

Done.


### 検証コード

In [4]:
import os
import re
import sys
import torch
from transformers import AutoModel

# ====== 設定 ======
HF_DIR = "/data3/taihei/matsunaga-repos/vhh_up/PLM/models/esm2_650m_base-sft_mean_ridge_tempro-vhh_weight_decay_search/encoder_weight_decay_0.1_head_weight_decay_0.001/encoder"
CONVERTED_PT = "/data3/taihei/matsunaga-repos/vhh_up/PLM/models/esm2_650m_base-sft_mean_ridge_tempro-vhh_weight_decay_search/encoder_weight_decay_0.1_head_weight_decay_0.001/encoder/converted_model.pt"  # 変換スクリプトの出力
CONVERTED_HAS_ESM_PREFIX = True    # 変換出力が 'esm.' プレフィクス付きなら True（推奨）
ATOL = 0.0                         # 完全一致をチェック。必要なら微小誤差を許容（例: 1e-7）
RTOL = 0.0

# ====== HF→ESM の確定マッピング（いただいたキー一覧に準拠） ======
LAYER_RE = re.compile(r"^encoder\.layer\.(\d+)\.")

def map_hf_key_to_esm(hf_key: str):
    """
    HF (.safetensors) のキーを ESM(.pt) のキーへマップ。
    戻り値: (esm_key, keep_flag)
      - esm_key: 出力 state_dict 上のキー（変換後 .pt のキーに合わせ 'esm.' 付きで返す）
      - keep_flag: False のとき比較対象外（例: position_embeddings）
    """
    m = LAYER_RE.match(hf_key)
    layer = int(m.group(1)) if m else None

    # Embeddings
    if hf_key == "embeddings.word_embeddings.weight":
        out = "embed_tokens.weight"
    elif hf_key == "embeddings.position_embeddings.weight":
        return None, False  # ESMはrotary; 位置埋め込みは捨てる

    # emb_layer_norm_after
    elif hf_key == "encoder.emb_layer_norm_after.weight":
        out = "emb_layer_norm_after.weight"
    elif hf_key == "encoder.emb_layer_norm_after.bias":
        out = "emb_layer_norm_after.bias"

    # contact head（HFにもある）
    elif hf_key == "contact_head.regression.weight":
        out = "contact_head.regression.weight"
    elif hf_key == "contact_head.regression.bias":
        out = "contact_head.regression.bias"

    # pooler -> head.0.*（SFT_hot.pt 準拠）
    elif hf_key == "pooler.dense.weight":
        out_full = "head.0.weight"   # SFT_hot.pt では 'esm.' なし
        return (out_full if not CONVERTED_HAS_ESM_PREFIX else out_full), True
    elif hf_key == "pooler.dense.bias":
        out_full = "head.0.bias"
        return (out_full if not CONVERTED_HAS_ESM_PREFIX else out_full), True

    # 各層
    elif layer is not None:
        if hf_key.endswith(".attention.self.rotary_embeddings.inv_freq"):
            out = f"layers.{layer}.self_attn.rot_emb.inv_freq"
        elif hf_key.endswith(".attention.self.query.weight"):
            out = f"layers.{layer}.self_attn.q_proj.weight"
        elif hf_key.endswith(".attention.self.query.bias"):
            out = f"layers.{layer}.self_attn.q_proj.bias"
        elif hf_key.endswith(".attention.self.key.weight"):
            out = f"layers.{layer}.self_attn.k_proj.weight"
        elif hf_key.endswith(".attention.self.key.bias"):
            out = f"layers.{layer}.self_attn.k_proj.bias"
        elif hf_key.endswith(".attention.self.value.weight"):
            out = f"layers.{layer}.self_attn.v_proj.weight"
        elif hf_key.endswith(".attention.self.value.bias"):
            out = f"layers.{layer}.self_attn.v_proj.bias"
        elif hf_key.endswith(".attention.output.dense.weight"):
            out = f"layers.{layer}.self_attn.out_proj.weight"
        elif hf_key.endswith(".attention.output.dense.bias"):
            out = f"layers.{layer}.self_attn.out_proj.bias"
        elif hf_key.endswith(".attention.LayerNorm.weight"):
            out = f"layers.{layer}.self_attn_layer_norm.weight"
        elif hf_key.endswith(".attention.LayerNorm.bias"):
            out = f"layers.{layer}.self_attn_layer_norm.bias"
        elif hf_key.endswith(".intermediate.dense.weight"):
            out = f"layers.{layer}.fc1.weight"
        elif hf_key.endswith(".intermediate.dense.bias"):
            out = f"layers.{layer}.fc1.bias"
        elif hf_key.endswith(".output.dense.weight"):
            out = f"layers.{layer}.fc2.weight"
        elif hf_key.endswith(".output.dense.bias"):
            out = f"layers.{layer}.fc2.bias"
        elif hf_key.endswith(".LayerNorm.weight"):
            out = f"layers.{layer}.final_layer_norm.weight"
        elif hf_key.endswith(".LayerNorm.bias"):
            out = f"layers.{layer}.final_layer_norm.bias"
        else:
            return None, False
    else:
        return None, False

    # 'esm.' プレフィクス付与（head.0.* は付けない）
    if out.startswith("layers.") or out.startswith("embed_") or out.startswith("emb_layer_norm_after") \
       or out.startswith("contact_head."):
        out_full = ("esm." + out) if CONVERTED_HAS_ESM_PREFIX else out
    else:
        out_full = out  # head.0.* はそのまま

    return out_full, True


# ====== 検証メイン ======
print("\n" + "="*50)
print("🔍 変換後 weight の厳密検証を開始します（lm_head.* 非許容）...")
print("="*50)

try:
    # --- HFロード ---
    hf_model = AutoModel.from_pretrained(HF_DIR, trust_remote_code=True)
    hf_sd = {k: v.cpu() for k, v in hf_model.state_dict().items()}

    # --- 変換PTロード ---
    conv_sd = torch.load(CONVERTED_PT, map_location="cpu")
    conv_keys = set(conv_sd.keys())
    print(f"HF params: {len(hf_sd)}  |  Converted PT params: {len(conv_sd)}")

    # ★ 先に、lm_head.* が含まれていないことを厳密チェック
    lm_head_in_conv = sorted([k for k in conv_keys if k.startswith("esm.lm_head.") or k.startswith("lm_head.")])
    if lm_head_in_conv:
        print("❌ エラー: 変換 .pt に 'lm_head.*' キーが含まれています。上書きの原因になるため非許容です。")
        for k in lm_head_in_conv[:20]:
            print("  -", k)
        sys.exit(1)

    # --- HF→ESM マッピングに基づき、値を1:1で比較 ---
    missing_in_converted, shape_mismatch, value_mismatch = [], [], []
    mapped_count = 0

    for hf_k, hf_v in hf_sd.items():
        esm_k, keep = map_hf_key_to_esm(hf_k)
        if not keep:
            continue
        mapped_count += 1

        if esm_k not in conv_sd:
            missing_in_converted.append((hf_k, esm_k))
            continue

        cv = conv_sd[esm_k].cpu()
        if hf_v.shape != cv.shape:
            shape_mismatch.append((hf_k, esm_k, tuple(hf_v.shape), tuple(cv.shape)))
            continue

        # 完全一致（必要であれば allclose）
        if ATOL == 0.0 and RTOL == 0.0:
            equal = torch.equal(hf_v, cv)
        else:
            equal = torch.allclose(hf_v, cv, atol=ATOL, rtol=RTOL)

        if not equal:
            max_abs = (hf_v - cv).abs().max().item()
            value_mismatch.append((hf_k, esm_k, max_abs))

    # --- 余剰キー（変換PTにあるがHFに由来しない）を洗う ---
    expected_from_hf = set()
    for hf_k in hf_sd.keys():
        esm_k, keep = map_hf_key_to_esm(hf_k)
        if keep and esm_k is not None:
            expected_from_hf.add(esm_k)

    extras = []
    for ck in conv_keys - expected_from_hf:
        # rotary 補完（ESM 参照から補った場合）だけは許容
        if ck.endswith(".rot_emb.inv_freq"):
            continue
        # pooler由来の head.0.* は許容（HF側 pooler のマッピング）
        if ck.startswith("head.0."):
            continue
        # ★ lm_head.* は許容しない（ここに来たら NG）
        if ck.startswith("esm.lm_head.") or ck.startswith("lm_head."):
            extras.append(ck)
            continue
        # その他の余剰は一応報告（通常は来ない想定）
        extras.append(ck)

    # --- 結果表示 ---
    print("\n--- 検証サマリ ---")
    print(f"Mapped (HF -> ESM keys): {mapped_count}")
    print(f"Missing in converted    : {len(missing_in_converted)}")
    print(f"Shape mismatch          : {len(shape_mismatch)}")
    print(f"Value mismatch          : {len(value_mismatch)}")
    print(f"Extras in converted     : {len(extras)}")

    if missing_in_converted:
        print("\n【Missingの例】HF -> ESM")
        for i, (h, e) in enumerate(missing_in_converted[:10], 1):
            print(f"  {i:02d}. {h}  ->  {e}")

    if shape_mismatch:
        print("\n【Shape mismatchの例】HF -> ESM (HF_shape / PT_shape)")
        for i, (h, e, s1, s2) in enumerate(shape_mismatch[:10], 1):
            print(f"  {i:02d}. {h} -> {e}  {s1} / {s2}")

    if value_mismatch:
        print("\n【Value mismatchの例】最大絶対誤差")
        for i, (h, e, d) in enumerate(value_mismatch[:10], 1):
            print(f"  {i:02d}. {h} -> {e}  max|Δ|={d:.3e}")

    if extras:
        print("\n【Extras（HFに由来しないのにPTに存在）】")
        for i, k in enumerate(extras[:20], 1):
            print(f"  {i:02d}. {k}")

    # --- 最終判定 ---
    print("\n--- 判定 ---")
    if (not missing_in_converted) and (not shape_mismatch) and (not value_mismatch) and (not extras):
        print("🎉 成功: 変換後 .pt は HF の重みを正確に反映しており、余計な lm_head.* も含まれていません。")
    else:
        print("⚠️ 注意: 差分があります。上の詳細を確認してマッピング/変換を見直してください。")

except Exception as e:
    print(f"検証中に予期せぬエラーが発生しました: {e}")


🔍 変換後 weight の厳密検証を開始します（lm_head.* 非許容）...
HF params: 569  |  Converted PT params: 568

--- 検証サマリ ---
Mapped (HF -> ESM keys): 568
Missing in converted    : 0
Shape mismatch          : 0
Value mismatch          : 0
Extras in converted     : 0

--- 判定 ---
🎉 成功: 変換後 .pt は HF の重みを正確に反映しており、余計な lm_head.* も含まれていません。
